# 02 - Crop Group Trends (Gold)

**Grain:** one row per year, quarter and crop group (PALAY, CORN, OTHER).
Year-over-year growth compares each quarter with the same quarter one year earlier.

Used by the dashboard.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from common.delta_io import write_delta_table
from configs.etl_config import SILVER_CROP_WEATHER_TABLE, GOLD_CROP_GROUP_TRENDS_TABLE


enriched_df = spark.table(SILVER_CROP_WEATHER_TABLE)

quarterly_totals_df = (
    enriched_df.groupBy("year", "quarter", "crop_group")
    .agg(
        F.round(F.sum("production"), 3).alias("total_production"),
        F.countDistinct("province").alias("province_count"),
    )
)

# Same crop group + same quarter, ordered by year -> the previous row is one year earlier
same_quarter_over_years = Window.partitionBy("crop_group", "quarter").orderBy("year")

trends_df = (
    quarterly_totals_df
    .withColumn("previous_year_production", F.lag("total_production", 1).over(same_quarter_over_years))
    .withColumn(
        "yoy_growth",
        F.when(
            F.col("previous_year_production") != 0,
            F.round(
                (F.col("total_production") - F.col("previous_year_production"))
                / F.col("previous_year_production"),
                4,
            ),
        ),
    )
)

write_delta_table(trends_df, GOLD_CROP_GROUP_TRENDS_TABLE)
spark.table(GOLD_CROP_GROUP_TRENDS_TABLE).groupBy("crop_group").count().orderBy("crop_group").show()